## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Xception

__Probado en:__ tensorflow 2.10.1
*****

Basada en la implementación: [Arjun Sarkar](https://medium.com/data-science/xception-from-scratch-using-tensorflow-even-better-than-inception-940fb231ced9)


## Librerias

In [ ]:
## Posibles librerias que necesitarían instalar, descomentar la siguiente linea
#!python -m pip install kagglehub opencv-python

In [ ]:
import kagglehub
from pathlib import Path
from glob import glob
import os
from time import time
import matplotlib.pyplot as plt

from PIL import Image 
from numpy import array, random, argmin, argmax
from pandas import DataFrame
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model
  
  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics 
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

## Diseño del modelo

In [ ]:
# creating the Conv-Batch Norm block
def conv_bn(x, filters, kernel_size, strides=1):
    """
        DESCRIIPTION:
            Traditional Convolution 2D + Batch Normalization layers.

        INPUT:
            @param x: feature maps tensor
            @type x: tf.tensor

            @param filters: number of convolutional filters
            @type filters: int

            @param kernel_size: kernel size filter
            @type kernel_size: int or tuple

            @param strides: stride step size
            @type strides: int

        OUTPUT:
            @param x: feature maps tensor
            @type x: tf.tensor

    """
    
    x = layers.Conv2D(filters=filters, 
                      kernel_size = kernel_size, 
                      strides=strides, 
                      padding = 'same', 
                      use_bias = False)(x)
    x = layers.BatchNormalization()(x)
    return x

# creating separableConv-Batch Norm block
def sep_bn(x, filters, kernel_size, strides=1):
    """
        DESCRIIPTION:
            Separable Convolution 2D + Batch Normalization layers.

        INPUT:
            @param x: feature maps tensor
            @type x: tf.tensor

            @param filters: number of convolutional filters
            @type filters: int

            @param kernel_size: kernel size filter
            @type kernel_size: int or tuple

            @param strides: stride step size
            @type strides: int

        OUTPUT:
            @param x: feature maps tensor
            @type x: tf.tensor

    """
    
    x = layers.SeparableConv2D(filters=filters, 
                               kernel_size = kernel_size, 
                               strides=strides, 
                               padding = 'same', 
                               use_bias = False)(x)
    x = layers.BatchNormalization()(x)
    return x

# entry flow
def entry_flow(x):
    """
        DESCRIPTION:
            Entry flow module of the Xception model 

        INPUT:
            @param x: Input tensor
            @type x: tf.tensor

        OUTPUT:
            @param x: Input tensor
            @type x: tf.tensor

    """
    x = conv_bn(x, filters =32, kernel_size =3, strides=2)
    x = layers.ReLU()(x)
    x = conv_bn(x, filters =64, kernel_size =3, strides=1)
    tensor = layers.ReLU()(x)
    
    ## Residual block 1
    x = sep_bn(tensor, filters = 128, kernel_size =3)
    x = layers.ReLU()(x)
    x = sep_bn(x, filters = 128, kernel_size =3)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding = 'same')(x)
    
    tensor = conv_bn(tensor, filters=128, kernel_size = 1,strides=2)
    x = layers.Add()([tensor,x])
    x = layers.ReLU()(x)

    ## Residual block 2
    x = sep_bn(x, filters =256, kernel_size=3)
    x = layers.ReLU()(x)
    x = sep_bn(x, filters =256, kernel_size=3)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding = 'same')(x)
    
    tensor = conv_bn(tensor, filters=256, kernel_size = 1,strides=2)
    x = layers.Add()([tensor,x])
    x = layers.ReLU()(x)

    ## Residual block 3
    x = sep_bn(x, filters =728, kernel_size=3)
    x = layers.ReLU()(x)
    x = sep_bn(x, filters =728, kernel_size=3)
    x = layers.MaxPool2D(pool_size=3, strides=2, padding = 'same')(x)
    
    tensor = conv_bn(tensor, filters=728, kernel_size = 1,strides=2)
    x = layers.Add()([tensor,x])

    ## return tensor
    return x

# middle flow
def middle_flow(tensor, loops=8):
    """
        DESCRIPTION:
            Middle flow module of the Xception model 

        INPUT:
            @param tensor: feature maps tensor
            @type tensor: tf.tensor

        OUTPUT:
            @param tensor: feature maps tensor
            @type tensor: tf.tensor
    """
    for _ in range(loops):
        x = layers.ReLU()(tensor)
        x = sep_bn(x, filters = 728, kernel_size = 3)
        x = layers.ReLU()(x)
        x = sep_bn(x, filters = 728, kernel_size = 3)
        x = layers.ReLU()(x)
        x = sep_bn(x, filters = 728, kernel_size = 3)
        x = layers.ReLU()(x)
        tensor = layers.Add()([tensor,x])
    
    ## return feature maps
    return tensor

def exit_flow(tensor, output_units=1000):
    """
        DESCRIPTION:
            Middle flow module of the Xception model 

        INPUT:
            @param tensor: feature maps tensor
            @type tensor: tf.tensor

            @param output_units: number of units in the output layer.
            @type output_units: int

        OUTPUT:
            @param tensor: feature maps tensor
            @type tensor: tf.tensor

    """
    x = layers.ReLU()(tensor)
    x = sep_bn(x, filters = 728,  kernel_size=3)
    x = layers.ReLU()(x)
    x = sep_bn(x, filters = 1024,  kernel_size=3)
    x = layers.MaxPool2D(pool_size = 3, strides = 2, padding ='same')(x)
    
    tensor = conv_bn(tensor, filters =1024, kernel_size=1, strides =2)
    x = layers.Add()([tensor,x])
    
    x = sep_bn(x, filters = 1536,  kernel_size=3)
    x = layers.ReLU()(x)
    x = sep_bn(x, filters = 2048,  kernel_size=3)
    x = layers.GlobalAvgPool2D()(x)
    
    if output_units == 1:
        x = layers.Dense(units = output_units, activation = 'sigmoid', name='output')(x)
    else:
        x = layers.Dense(units = output_units, activation = 'softmax', name='output')(x)
    
    ## return tensor
    return x

def Xception(input_shape, output_units=1000):
    """
        DESCRIPTION:
            Xception model builder

        INPUT:
            @param input_shape
            @type input_shape

            @param output_units
            @type output_units

        OUTPUT:
            @param model
            @type model
    """
    ## Flujo de operaciones
    input = layers.Input(shape = input_shape)
    x = entry_flow(input)
    x = middle_flow(x)
    output = exit_flow(x, output_units)

    ## Construcción del modelo
    model = Model (inputs=input, outputs=output, name='Xception')

    ## retorno del modelo 
    return model

## Dataset

<center>
    <img src=https://miro.medium.com/v2/resize:fit:1100/format:webp/1*djNpq2YLrbBGiWfy_DB7OQ.jpeg width=800>
</center>

Fuente de datos: [Garbage Classification](https://www.kaggle.com/datasets/mostafaabla/garbage-classification/data)

Este conjunto de datos contiene 15.150 imágenes de 12 clases diferentes de basura doméstica: papel, cartón, biológico, metal, plástico, vidrio verde, vidrio marrón, vidrio blanco, ropa, zapatos, pilas y basura.

**Objetivo**: Clasificar las imágenes según su clase. 


#### Carga de datos

In [ ]:
## Descarga del dataset desde kaggle
path = kagglehub.dataset_download("mostafaabla/garbage-classification")

## Declaración de la ruta de los datos
src = Path(path)
folder = Path.joinpath(src, 'garbage_classification')
print("Path to dataset files:", folder)

## Armar un dataframe con las rutas de las imágenes y su respectiva clase 
files = glob(f'{folder}/**/*.jpg', recursive=True)
dataset = DataFrame({'file': files})
if os.name == 'posix':
    dataset['class'] = dataset['file'].apply(lambda x: x.split('/')[-2])
else:
    dataset['class'] = dataset['file'].apply(lambda x: x.split('\\')[-2])

dataset.head()

#### Visualización de algunos datos

In [ ]:
random.seed(0)
n_list = random.choice(a=range(len(dataset)), size=16, replace=False)

plt.figure(figsize=(8, 8))
for i, j in enumerate(n_list):
    file, c_name = dataset.iloc[j]
    image = Image.open(file)
    image = image.resize((100, 100))
    image = array(image)

    plt.subplot(4, 4, i+1)
    plt.imshow(image)
    plt.title(c_name)
    plt.axis('off')
    
plt.tight_layout()
plt.show()

In [ ]:
dataset['class'].value_counts()

#### Preprocesamiento de datos y generadores

In [ ]:
## Partición de los datos 
train_data, test_data = train_test_split(dataset, stratify=dataset['class'], 
                                         test_size=0.2)

print('Train (shape) {}'.format(train_data.shape))
print('Test (shape) {}'.format(test_data.shape))

In [ ]:
target_size = (299, 299)
batch_size=32

## Instancia de los generadores
trainVal_generator_instance = ImageDataGenerator(rescale=1/255.0,
                                        validation_split=0.2,
                                        )

test_generator_instance = ImageDataGenerator(rescale=1/255.0)

## Configuración de los generadores
train_generator = trainVal_generator_instance.flow_from_dataframe(dataframe=train_data,
                                                                  x_col='file', y_col='class',
                                                                  target_size=target_size,
                                                                  color_mode='rgb',
                                                                  class_mode='categorical',
                                                                  batch_size=batch_size,
                                                                  shuffle=True,
                                                                  subset='training')

val_generator = trainVal_generator_instance.flow_from_dataframe(dataframe=train_data,
                                                                x_col='file', y_col='class',
                                                                target_size=target_size,
                                                                color_mode='rgb',
                                                                class_mode='categorical',
                                                                batch_size=batch_size,
                                                                shuffle=True,
                                                                subset='validation')

test_generator = test_generator_instance.flow_from_dataframe(dataframe=test_data,
                                                             x_col='file', y_col='class',
                                                             target_size=target_size,
                                                             color_mode='rgb',
                                                             class_mode='categorical',
                                                             batch_size=batch_size,
                                                             shuffle=False)


## Modelo

In [ ]:
model = Xception(input_shape=(*target_size, 3), output_units=12)
model.summary()

#### Ajuste del modelo

In [ ]:
## Tiempo estimado de ejecución: 124 minutos ~ 2hrs (GPU)
start = time()

## Configuración del modelo
model.compile(loss='categorical_crossentropy', 
              optimizer='adam', 
              metrics=['accuracy'])

## Ajuste del modelo
history = model.fit(train_generator, validation_data=val_generator, epochs=50)

timeUp = time()
print('Time spent[s]: {:.3f}'.format(timeUp - start))

In [ ]:
## Grafica de desempeño
plot_history(history, width=12, height=12)

In [ ]:
## Searching for the best epoch
id_min = argmin(history.history['val_loss'])
print('Loss - Validation: {} - Error: {}'.format(id_min+1, history.history['val_loss'][id_min]))

id_max = argmax(history.history['val_accuracy'])
print('Accuracy - Validation: {} - Error: {}'.format(id_max+1, history.history['val_accuracy'][id_max]))

## Mejor modelo

In [ ]:
train_generator_instance = ImageDataGenerator(rescale=1/255.0)

## Configuración de los generadores
train_generator = train_generator_instance.flow_from_dataframe(dataframe=train_data,
                                                               x_col='file', y_col='class',
                                                               target_size=target_size,
                                                               color_mode='rgb',
                                                               class_mode='categorical',
                                                               batch_size=batch_size,
                                                               shuffle=True
                                                               )

## Instancia del modelo
model = Xception(input_shape=(*target_size, 3), output_units=12)

## Tiempo estimado de ejecución: 141 minutos ~ 2.30hrs (GPU)
start = time()

## Configuración del modelo
model.compile(loss='categorical_crossentropy', 
              optimizer='adam', 
              metrics=['accuracy'])

## Ajuste del modelo
model.fit(train_generator, epochs=48)

timeUp = time()
print('Time spent[s]: {:.3f}'.format(timeUp - start))

#### Computo de predicción

Modelo pre-entrenado: [Descarga](https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/anthony_cho_l_edu_uai_cl/ESuMLSrYdX1AvfDpcW4UWd0Bl634s0aZ1AgpkKQ00y9pZg?download=1)

In [ ]:
## Descarga del modelo pre-entrenado
if os.name == 'posix':
    !wget 'https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/anthony_cho_l_edu_uai_cl/ESuMLSrYdX1AvfDpcW4UWd0Bl634s0aZ1AgpkKQ00y9pZg?download=1' 
    
    ## Rename file
    !mv ESuMLSrYdX1AvfDpcW4UWd0Bl634s0aZ1AgpkKQ00y9pZg?download=1 "Xception_best.h5"


In [ ]:
test_generator.reset()

## Predicciones
predicciones = model.predict(test_generator)
print('(shape) predicciones: {}'.format(predicciones.shape))

## Decodificación de las predicciones
prediccion_labels = predicciones.argmax(axis=1)

## Etiquetas reales
y_labels = test_generator.labels

In [ ]:
## Display confusion matrix
cm = confusion_matrix(y_pred=prediccion_labels, y_true=y_labels)
CM = ConfusionMatrixDisplay(confusion_matrix=cm)
CM.plot()
plt.show()

In [ ]:
## Display classification report
print(classification_report(y_pred=prediccion_labels, 
                            y_true=y_labels, 
                            digits=2) )

In [ ]:
index_names_dict = {value: key for key, value in test_generator.class_indices.items()}
index_names_dict

In [ ]:
#random.seed(0)
n_list = random.choice(a=range(len(test_data)), size=16, replace=False)

plt.figure(figsize=(8, 8))
for i, j in enumerate(n_list):
    file, c_name = test_data.iloc[j]
    pred_name = index_names_dict[prediccion_labels[j]]
    image = Image.open(file)
    image = image.resize((100, 100))
    image = array(image)

    plt.subplot(4, 4, i+1)
    plt.imshow(image)
    plt.title('True: {}\nPred: {}'.format(c_name, pred_name))
    plt.axis('off')
    
plt.tight_layout()
plt.show()